In [1]:
import sys
import warnings
import numpy as np

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
from scipy.stats import norm
sys.path.insert(0, "../../Utilities/")
import common_functions as cf

warnings.filterwarnings('ignore')

In [2]:
data_in = np.load('initial_inputs.npy')
data_out = np.load('initial_outputs.npy')
#print(data_in)
#print(data_out)

Week-01

In [3]:
X_init = np.array([
[0.7281861,  0.15469257, 0.73255167, 0.69399651, 0.05640131],
[0.24238435, 0.84409997, 0.5778091 , 0.67902128, 0.50195289],
[0.72952261, 0.7481062 , 0.67977464, 0.35655228, 0.67105368],
[0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
[0.6188123 , 0.33180214, 0.18728787, 0.75623847, 0.3288348 ],
[0.78495809, 0.91068235, 0.7081201 , 0.95922543, 0.0049115 ],
[0.14511079, 0.8966846 , 0.89632223, 0.72627154, 0.23627199],
[0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
[0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
[0.75759436, 0.35583141, 0.0165229 , 0.4342072 , 0.11243304],
[0.5367969 , 0.30878091, 0.41187929, 0.38822518, 0.5225283 ],
[0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
[0.6293079 , 0.80348368, 0.81140844, 0.04561319, 0.11062446],
[0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
[0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
[0.25890557, 0.79367771, 0.6421139 , 0.19667346, 0.59310318],
[0.43216593, 0.71561781, 0.3418191 , 0.70499988, 0.61496184],
[0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
[0.9217762 , 0.93187122, 0.41487637, 0.59505727, 0.73562569],
[0.12667892, 0.2914703 , 0.06452848, 0.6805146 , 0.89281919]
])
 

y_init = np.array([
-0.71426495, -1.20995524, -1.67219994, -1.53605771, -0.82923655, -1.24704893,
-1.23378638, -1.69434344, -2.57116963, -1.30911635, -1.14478485, -1.91267714,
-1.62283895, -1.35668211, -2.0184254 , -1.70255784, -1.29424696, -0.93575656,
-2.15576776, -1.74688209
])

X_init = data_in
y_init = data_out

y_transformed = -y_init
y_best = y_transformed.max()

kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)


GaussianProcessRegressor(alpha=1e-06, kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)

In [4]:
# --- Step 3: Define Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because we will minimize

In [6]:
# --- Step 4: Optimize acquisition function ---
bounds = [(0,1)]*5  # 5 ingredients
best_x = None
best_ei = float('inf')

# Multiple random starts to avoid local maxima
for _ in range(20):
    x0 = np.random.rand(5)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print(cf.format_inputdata(x_next))


0.047020-0.934958-0.399168-0.017548-0.480228


Week-02

In [7]:
datain = data_in
dataout = data_out

x_new = np.array([0.314365, 0.000000, 0.399314, 1.000000, 0.000000])
y_new = np.array([-1.1000925633939322])

X_init = np.vstack([datain, x_new])
y_init = np.hstack([dataout, y_new])

print(X_init)
print(y_init)

[[0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.72952261 0.7481062  0.67977464 0.35655228 0.67105368]
 [0.77062024 0.11440374 0.04677993 0.64832428 0.27354905]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.78495809 0.91068235 0.7081201  0.95922543 0.0049115 ]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.94506907 0.28845905 0.97880576 0.96165559 0.59801594]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.75759436 0.35583141 0.0165229  0.4342072  0.11243304]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.95773967 0.23566857 0.09914585 0.15680593 0.07131737]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.70499988 0.61496184]
 [0.78287982 0

In [12]:
# --- Step 1: Transform outputs for maximization ---
y_transformed = -y_init   # since we want to make score close to zero (maximise negative of total)
y_best = y_transformed.max()

# --- Step 2: Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_transformed)

# --- Step 3: Define acquisition functions ---

# Expected Improvement (EI)
def expected_improvement(x, gp, y_best, xi=0.05):  # adjust xi for exploration/exploitation balance
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # we minimize in scipy

# --- (Optional) UCB version: uncomment to use ---
# def ucb(x, gp, kappa=2.0):
#     x = np.array(x).reshape(1, -1)
#     mu, sigma = gp.predict(x, return_std=True)
#     return -(mu + kappa * sigma)  # negative since we minimize

# --- Step 4: Optimize acquisition function ---
bounds = [(0, 1)] * 5
best_x = None
best_val = float('inf')

for _ in range(40):  # more restarts to reduce local optima risk
    x0 = np.random.rand(5)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    # --- For UCB, replace expected_improvement(...) with ucb(...)
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x
print(cf.format_inputdata(best_x))
#print(cf.format_inputdata(x_next))

0.120977-0.479385-0.259477-0.487875-0.900965
